# Measurements preparations

## Preamble

In [2]:
import os
from glob import glob
from datetime import datetime
import csv
import xml.etree.ElementTree as ET


import pandas as pd
from libnmap.parser import NmapParser
import ipaddress

## IPv4

### NMap allowlist

In [59]:
ts = datetime(2025, 8, 27)  # date when ISI hitlist was made

In [60]:
_anycast_ipv4_pdf = pd.read_csv(f"anycast_census_{ts.year}_{ts.month:02d}_{ts.day:02d}_v4.csv")
anycast_ipv4_pdf = _anycast_ipv4_pdf[_anycast_ipv4_pdf["GCD_ICMPv4"] > 1].copy()
display(anycast_ipv4_pdf.head(5))
print(len(anycast_ipv4_pdf))

,prefix,AB_ICMPv4,AB_TCPv4,AB_DNSv4,GCD_ICMPv4,GCD_TCPv4,partial,backing_prefix,ASN,locations
0,1.0.0.0/24,29,29,29,75,30,False,1.0.0.0/24,13335,"[{'city': 'Honolulu', 'code_country': 'US', 'i..."
1,1.1.1.0/24,29,29,0,74,31,False,1.1.1.0/24,13335,"[{'city': 'Bangalore', 'code_country': 'IN', '..."
3,1.10.10.0/24,2,0,2,2,0,False,1.10.10.0/24,148000,"[{'city': 'Mumbai', 'code_country': 'IN', 'id'..."
4,1.12.0.0/24,5,0,5,5,0,False,1.12.0.0/20,132203_45090,"[{'city': 'Cologne', 'code_country': 'DE', 'id..."
5,1.12.12.0/24,3,1,0,3,1,False,1.12.0.0/20,132203_45090,"[{'city': 'Hong Kong', 'code_country': 'HK', '..."


13361


In [61]:
ipv4_pdf = pd.read_csv(f"internet_address_verfploeter_hitlist_it113w-{ts.year}{ts.month:02d}{ts.day:02d}.csv", header=None)
ipv4_pdf.columns = ["ipv4"]

display(ipv4_pdf.head(5))
print(len(ipv4_pdf))

,ipv4
0,1.0.0.0
1,1.0.4.1
2,1.0.5.1
3,1.0.6.1
4,1.0.7.1


5788894


In [62]:
ipv4_pdf["prefix"] = ipv4_pdf["ipv4"].apply(lambda x: ".".join(x.split(".")[:3]) + ".0/24")

display(ipv4_pdf.head(5))

,ipv4,prefix
0,1.0.0.0,1.0.0.0/24
1,1.0.4.1,1.0.4.0/24
2,1.0.5.1,1.0.5.0/24
3,1.0.6.1,1.0.6.0/24
4,1.0.7.1,1.0.7.0/24


In [63]:
responsive_ipv4_pdf = anycast_ipv4_pdf.merge(ipv4_pdf, on="prefix", how="inner")

display(responsive_ipv4_pdf.head(5))
print(len(responsive_ipv4_pdf))

,prefix,AB_ICMPv4,AB_TCPv4,AB_DNSv4,GCD_ICMPv4,GCD_TCPv4,partial,backing_prefix,ASN,locations,ipv4
0,1.0.0.0/24,29,29,29,75,30,False,1.0.0.0/24,13335,"[{'city': 'Honolulu', 'code_country': 'US', 'i...",1.0.0.0
1,1.1.1.0/24,29,29,0,74,31,False,1.1.1.0/24,13335,"[{'city': 'Bangalore', 'code_country': 'IN', '...",1.1.1.0
2,1.10.10.0/24,2,0,2,2,0,False,1.10.10.0/24,148000,"[{'city': 'Mumbai', 'code_country': 'IN', 'id'...",1.10.10.10
3,1.12.0.0/24,5,0,5,5,0,False,1.12.0.0/20,132203_45090,"[{'city': 'Cologne', 'code_country': 'DE', 'id...",1.12.0.0
4,1.12.12.0/24,3,1,0,3,1,False,1.12.0.0/20,132203_45090,"[{'city': 'Hong Kong', 'code_country': 'HK', '...",1.12.12.12


13355


In [71]:
responsive_ipv4 = responsive_ipv4_pdf.drop_duplicates(subset=["ipv4"])["ipv4"].copy()
responsive_ipv4.to_csv(f"input/nmap/responsive_anycast_ipv4_addresses_{ts.year}{ts.month:02d}{ts.day:02d}.csv", index=False, header=False)

In [ ]:
# dividing into shards to run nmap in parallel if necessary
nshards = 30

for shard in range(nshards):
    shard_pdf = responsive_ipv4.iloc[shard::nshards]
    shard_pdf.to_csv(f"input/nmap/sharding/responsive_anycast_ipv4_addresses_shard_{shard+1}_of_{nshards}_{ts.year}{ts.month:02d}{ts.day:02d}.csv", index=False, header=False)

In [ ]:
# confirming that shards don't overlap

all_ips = set()
for shard in range(nshards):
    shard_pdf = pd.read_csv(f"input/nmap/sharding/responsive_anycast_ipv4_addresses_shard_{shard+1}_of_{nshards}_{ts.year}{ts.month:02d}{ts.day:02d}.csv", header=None)
    shard_pdf.columns = ["ipv4"]
    shard_ips = set(shard_pdf["ipv4"].tolist())
    intersection = all_ips.intersection(shard_ips)
    assert len(intersection) == 0, f"Overlap found in shard {shard+1}, {intersection}"
    all_ips.update(shard_ips)

### ZMap allowlist

In [35]:
ts = datetime(2025, 11, 26)

In [ ]:
_anycast_ipv4_pdf = pd.read_csv(f"anycast_census_{ts.year}_{ts.month:02d}_{ts.day:02d}_v4.csv")
anycast_ipv4_pdf = _anycast_ipv4_pdf[_anycast_ipv4_pdf["GCD_ICMPv4"] > 1].copy()
display(anycast_ipv4_pdf.head(5))
print(len(anycast_ipv4_pdf))
anycast_ipv4_pdf["prefix"].to_csv(f"input/anycast_prefixes_{ts.year}_{ts.month:02d}_{ts.day:02d}_v4.csv", index=False, header=False)

,prefix,AB_ICMPv4,AB_TCPv4,AB_DNSv4,GCD_ICMPv4,GCD_TCPv4,partial,backing_prefix,ASN,locations
0,1.0.0.0/24,28,29,28,66,30,False,1.0.0.0/24,13335,"[{'city': 'Bangalore', 'country_code': 'IN', '..."
1,1.1.1.0/24,28,29,24,66,30,False,1.1.1.0/24,13335,"[{'city': 'Bangalore', 'country_code': 'IN', '..."
3,1.10.10.0/24,3,0,3,2,0,False,1.10.10.0/24,148000,"[{'city': 'Delhi', 'country_code': 'IN', 'id':..."
4,1.12.0.0/24,5,0,5,5,0,False,1.12.0.0/20,132203,"[{'city': 'Singapore', 'country_code': 'SG', '..."
5,1.12.12.0/24,3,1,0,3,1,False,1.12.0.0/20,132203,"[{'city': 'Hong Kong', 'country_code': 'HK', '..."


14010


#### ZMap ports

In [ ]:
xml_files = glob("results/nmap/*.xml")

ports = set()
for xml_file in xml_files:
    print(f"Processing {xml_file}...")
    report = NmapParser.parse_fromfile(xml_file)
    for host in report.hosts:
        for svc in host.services:
            if svc.state == "open":
                ports.add(svc.port)


print(f"Discovered {len(ports)} open ports")

with open("input/zmap/nmap_open_ports.txt", "w") as f:
    for item in ports:
        f.write(f"{item}\n")

Processing results/nmap/output.xml...
Processing results/nmap/nmap_20251201133002_.xml...
Discovered 997 open ports


### ZGrab2 allowlist

In [5]:
def store_zgrab_input(dataset):
    # zmap v4 files are like: zmap_465_20251127123220.csv
    zmap_files = glob(f"../results/zmap/dataset={dataset}/**/zmap_*.csv", recursive=True)

    zmap_pdf = pd.DataFrame(columns=["saddr", "port"])
    for zmap_file in zmap_files:
        # extracting the port from the file name
        port = int(zmap_file.split("_")[-2])
        zmap_port_pdf = pd.read_csv(zmap_file)
        zmap_port_pdf["port"] = port
        zmap_pdf = pd.concat([zmap_pdf, zmap_port_pdf[["saddr", "port"]]], ignore_index=True)

    zmap_pdf["domain"] = ""
    zmap_pdf["tag"] = ""
    zmap_pdf[["saddr", "domain", "tag", "port"]].to_csv(f"../input/zgrab/zgrab_input_{dataset}.csv", index=False, header=False)

store_zgrab_input("tcp-anycast")

27


### NMap allowlist -- old pipeline

In [44]:
def create_allowlist_from_zmap(dataset, ts, output_filename):
    # zmap v4 files are like: zmap_465_20251127123220_v4.csv
    zmap_v4_files = glob(f"results/zmap/dataset={dataset}/**/zmap_*.csv", recursive=True)

    zmap_v4_pdf = pd.DataFrame(columns=["saddr", "port"])
    for zmap_v4_file in zmap_v4_files:
        # extracting the port from the file name
        port = int(zmap_v4_file.split("_")[-2])
        zmap_port_v4_pdf = pd.read_csv(zmap_v4_file)
        zmap_port_v4_pdf["port"] = port
        zmap_v4_pdf = pd.concat([zmap_v4_pdf, zmap_port_v4_pdf[["saddr", "port"]]], ignore_index=True)

    zmap_v4_pdf.drop_duplicates(["saddr"])["saddr"].to_csv(f"input/nmap/{output_filename}_{ts.year}{ts.month:02d}{ts.day:02d}.csv", index=False, header=False)

In [ ]:
ts = datetime(2025, 11, 28)  # date when ZMap scan was done
create_allowlist_from_zmap("tcp-anycast", ts, "zmap_responsive_tcp_ipv4_addresses")

### QUIC allowlist

In [45]:
ts = datetime(2025, 11, 28)  # date when ZMap UDP scan was done
create_allowlist_from_zmap("udp-anycast", ts, "zmap_responsive_udp_ipv4_addresses")

## IPv6

### ZMap allowlist -- unused

In [2]:
ts_v6 = datetime(2025, 11, 26)

In [ ]:
_anycast_ipv6_pdf = pd.read_csv(f"anycast_census_{ts_v6.year}_{ts_v6.month:02d}_{ts_v6.day:02d}_v6.csv")
anycast_ipv6_pdf = _anycast_ipv6_pdf[_anycast_ipv6_pdf["GCD_ICMPv6"] > 1].copy()
display(anycast_ipv6_pdf.head(5))
print(len(anycast_ipv6_pdf))
anycast_ipv6_pdf["prefix"].to_csv(f"input/anycast_prefixes_{ts_v6.year}_{ts_v6.month:02d}_{ts_v6.day:02d}_v6.csv", index=False, header=False)

,prefix,AB_ICMPv6,AB_TCPv6,AB_DNSv6,GCD_ICMPv6,GCD_TCPv6,backing_prefix,ASN,locations
0,2001:1201::/48,2,2,2,2,1,2001:1201::/48,28498,"[{'city': 'Querétaro', 'country_code': 'MX', '..."
11,2001:1258::/48,2,2,2,2,2,2001:1258::/32,28499,"[{'city': 'Querétaro', 'country_code': 'MX', '..."
13,2001:12f8:2::/48,3,0,3,3,0,2001:12f8:2::/48,12136,"[{'city': 'Singapore', 'country_code': 'SG', '..."
14,2001:12f8:4::/48,4,0,4,5,0,2001:12f8:4::/48,11644,"[{'city': None, 'country_code': None, 'id': 'N..."
15,2001:12f8:8::/48,3,0,3,3,0,2001:12f8:8::/48,10906,"[{'city': 'St Petersburg', 'country_code': 'RU..."


12821


### NMap allowlist

In [9]:
ts_v6 = datetime(2025, 8, 18)

In [10]:
_anycast_ipv6_pdf = pd.read_csv(f"anycast_census_{ts_v6.year}_{ts_v6.month:02d}_{ts_v6.day:02d}_v6.csv")
anycast_ipv6_pdf = _anycast_ipv6_pdf[_anycast_ipv6_pdf["GCD_ICMPv6"] > 1].copy()
display(anycast_ipv6_pdf.head(5))
print(len(anycast_ipv6_pdf))

,prefix,AB_ICMPv6,AB_TCPv6,AB_DNSv6,GCD_ICMPv6,GCD_TCPv6,backing_prefix,ASN,locations
0,2001:1201:10::/48,1,1,1,3,1,2001:1201:10::/48,27661,"[{'city': 'Vancouver', 'code_country': 'CA', '..."
1,2001:1201::/48,2,2,2,3,2,2001:1201::/48,28498,"[{'city': 'Monterrey', 'code_country': 'MX', '..."
6,2001:1250:a000::/48,3,2,3,2,1,2001:1250:a000::/44,28511_22894,"[{'city': 'Monterrey', 'code_country': 'MX', '..."
8,2001:1250:c000::/48,3,2,3,2,1,2001:1250:c000::/44,28511_22894,"[{'city': 'Monterrey', 'code_country': 'MX', '..."
10,2001:1258::/48,2,2,2,3,2,2001:1258::/32,28499,"[{'city': 'Querétaro', 'code_country': 'MX', '..."


6019


In [11]:
ipv6_pdf = pd.read_csv(f"v6_{ts_v6.year}{ts_v6.month:02d}{ts_v6.day:02d}.csv", header=None)

ipv6_pdf.columns = ["ipv6"]
display(ipv6_pdf.head(5))
print(len(ipv6_pdf))

,ipv6
0,2620:10a:80ac::45
1,2001:678:2c:0:194:0:28:7
2,2001:678:78:42:ad::53
3,2001:dce:2000:2::130
4,2001:dce:7000:2::130


6234585


In [12]:
def is_v6_in_prefix(ipv6, prefix):
    try:
        ipv6_addr = ipaddress.IPv6Address(ipv6)
        ipv6_net = ipaddress.IPv6Network(prefix)
        return ipv6_addr in ipv6_net
    except ValueError:
        return False


def ipv6_to_prefix48(ip):
    try:
        # Create a /48 network that contains this IP, non-strict so host bits allowed
        net = ipaddress.IPv6Network(f"{ip}/48", strict=False)
        # Return in same format as your prefix column, e.g. "2001:1201:10::/48"
        return net.with_prefixlen
    except ValueError:
        return None


ipv6_pdf["prefix"] = ipv6_pdf["ipv6"].apply(lambda x: ipv6_to_prefix48(x))
display(ipv6_pdf.head(5))

,ipv6,prefix
0,2620:10a:80ac::45,2620:10a:80ac::/48
1,2001:678:2c:0:194:0:28:7,2001:678:2c::/48
2,2001:678:78:42:ad::53,2001:678:78::/48
3,2001:dce:2000:2::130,2001:dce:2000::/48
4,2001:dce:7000:2::130,2001:dce:7000::/48


In [13]:
responsive_ipv6_pdf = anycast_ipv6_pdf.merge(ipv6_pdf, on="prefix", how="inner")

display(responsive_ipv6_pdf.head(5))
print(len(responsive_ipv6_pdf))

,prefix,AB_ICMPv6,AB_TCPv6,AB_DNSv6,GCD_ICMPv6,GCD_TCPv6,backing_prefix,ASN,locations,ipv6
0,2001:1201:10::/48,1,1,1,3,1,2001:1201:10::/48,27661,"[{'city': 'Vancouver', 'code_country': 'CA', '...",2001:1201:10::1
1,2001:1201::/48,2,2,2,3,2,2001:1201::/48,28498,"[{'city': 'Monterrey', 'code_country': 'MX', '...",2001:1201::1
2,2001:1250:a000::/48,3,2,3,2,1,2001:1250:a000::/44,28511_22894,"[{'city': 'Monterrey', 'code_country': 'MX', '...",2001:1250:a000::1
3,2001:1250:c000::/48,3,2,3,2,1,2001:1250:c000::/44,28511_22894,"[{'city': 'Monterrey', 'code_country': 'MX', '...",2001:1250:c000::1
4,2001:1258::/48,2,2,2,3,2,2001:1258::/32,28499,"[{'city': 'Querétaro', 'code_country': 'MX', '...",2001:1258::1


6017


In [14]:
responsive_ipv6_pdf["ipv6"].to_csv(f"input/responsive_anycast_ipv6_addresses_{ts_v6.year}{ts_v6.month:02d}{ts_v6.day:02d}.csv", index=False, header=False)

# Process output

## Process ZMap output

In [ ]:
def process_zmap_anycast_results(dataset):
    # zmap v4 files are like: zmap_465_20251127123220.csv
    zmap_v4_files = glob(f"results/zmap/dataset={dataset}/**/zmap_*.csv", recursive=True)

    zmap_v4_pdf = pd.DataFrame(columns=["saddr", "port"])
    for zmap_v4_file in zmap_v4_files:
        # extracting the port from the file name
        port = int(zmap_v4_file.split("_")[-2])
        zmap_port_v4_pdf = pd.read_csv(zmap_v4_file)
        zmap_port_v4_pdf["port"] = port
        zmap_v4_pdf = pd.concat([zmap_v4_pdf, zmap_port_v4_pdf[["saddr", "port"]]], ignore_index=True)

    display(
        zmap_v4_pdf.groupby("port").size().reset_index(name="counts").sort_values(by="counts", ascending=False)
    )

    print("unique reponsive IPv4 addresses:", zmap_v4_pdf["saddr"].nunique())

process_zmap_anycast_results("tcp-anycast")

,port,counts
8,443,2428483
4,80,2376009
3,53,683857
5,110,623389
14,995,619655
13,993,619218
1,22,614430
2,25,610767
19,3306,609978
6,143,609429


unique reponsive IPv4 addresses: 2598785


In [ ]:
process_zmap_anycast_results("udp-anycast")

,port,counts
0,443,1053948
1,853,1327


unique reponsive IPv4 addresses: 1053974


## Process nmap output

In [57]:
xml_files = glob("results/nmap/*.xml")

for xml_file in xml_files:
    print(f"Processing {xml_file}...")

    # parse rtt for each host
    tree = ET.parse(xml_file)
    root = tree.getroot()

    host_rtts = {}
    for host in root.findall("host"):
        addr_el = host.find("address")
        ip = addr_el.get("addr") if addr_el is not None else None

        times = host.find("times")
        if times is not None:
            srtt = int(times.get("srtt")) / 1000   # convert to ms
            host_rtts[ip] = srtt

    report = NmapParser.parse_fromfile(xml_file)
    filename = os.path.splitext(os.path.basename(xml_file))[0]
    with open(f"results/nmap/{filename}.csv", "wt", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["host", "rtt_ms", "port", "protocol", "service", "os", "banner"])
        for host in report.hosts:
            #print(dir(host))
            for svc in host.services:
                writer.writerow([
                    host.address,
                    host_rtts.get(host.address),
                    svc.port,
                    svc.protocol,
                    svc.service,
                    host.os,
                    svc.banner,
                ])



Processing results/nmap/nmap_20251127155539_.xml...
Processing results/nmap/nmap_20251128195856_.xml...
Processing results/nmap/nmap_20251128203007_.xml...
Processing results/nmap/output.xml...
Processing results/nmap/nmap_20251128193545_.xml...


In [21]:
# https://secwiki.org/w/FAQ_tcpwrapped

In [58]:
nmap_pdf = pd.read_csv("results/nmap/nmap_20251128203007_.csv")
display(nmap_pdf)
print(len(nmap_pdf))

nmap_pdf.groupby(["host", "service"]).size().reset_index(name="counts").sort_values(by="counts", ascending=False)

,host,rtt_ms,port,protocol,service,os,banner


0


,host,service,counts


In [49]:
nmap_pdf = pd.read_csv("results/nmap/output.csv")
display(nmap_pdf)
print(len(nmap_pdf))

,host,rtt_ms,port,protocol,service,os,banner
0,1.0.0.0,1.212,80,tcp,http,Fingerprints:,product: Cloudflare http proxy
1,1.0.0.0,1.212,443,tcp,https,Fingerprints:,product: cloudflare
2,1.0.0.0,1.212,8080,tcp,http,Fingerprints:,product: Cloudflare http proxy
3,1.0.0.0,1.212,8443,tcp,https-alt,Fingerprints:,product: cloudflare
4,1.1.1.0,1.250,80,tcp,http,Fingerprints:,product: Cloudflare http proxy
5,1.1.1.0,1.250,443,tcp,https,Fingerprints:,product: cloudflare
6,1.1.1.0,1.250,8080,tcp,http,Fingerprints:,product: Cloudflare http proxy
7,1.1.1.0,1.250,8443,tcp,https-alt,Fingerprints:,product: cloudflare
8,1.1.8.8,207.842,80,tcp,http,Fingerprints:,product: nginx version: 1.20.1
9,1.1.8.8,207.842,443,tcp,http,Fingerprints:,product: OpenResty web app server


24


## Process mtr output

In [34]:
mtr_files = glob("results/mtr/*.csv")
for mtr_file in mtr_files:
    print(f"Processing {mtr_file}...")
    # Add your processing code here
    mtr_pdf = pd.read_csv(mtr_file)
    display(mtr_pdf)

Processing results/mtr/mtr_1.12.15.31.csv...


,hostname,Mtr_Version,Start_Time,Status,Host,Hop,Ip,Asn,Loss%,Snt,,Last,Avg,Best,Wrst,StDev
0,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,1,172.18.0.1 (172.18.0.1),AS???,0.0,1,0,0.33,0.33,0.33,0.33,0.0
1,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,2,169.254.169.254 (169.254.169.254),AS???,0.0,1,0,0.20,0.20,0.20,0.20,0.0
2,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,3,vl199-ds1-j2-51917.ams1.constant.com (45.76.40...,AS20473,0.0,1,0,18.60,18.60,18.60,18.60,0.0
3,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,4,10.72.0.5 (10.72.0.5),AS???,0.0,1,0,1.78,1.78,1.78,1.78,0.0
4,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,5,10.72.4.13 (10.72.4.13),AS???,0.0,1,0,0.95,0.95,0.95,0.95,0.0
5,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,6,5-1-40.ear3.amsterdam1.level3.net (213.19.196....,AS3356,0.0,1,0,1.63,1.63,1.63,1.63,0.0
6,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,7,???,AS???,100.0,1,1,0.00,0.00,0.00,0.00,0.0
7,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,8,62.67.67.70 (62.67.67.70),AS3356,0.0,1,0,8.88,8.88,8.88,8.88,0.0
8,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,9,???,AS???,100.0,1,1,0.00,0.00,0.00,0.00,0.0
9,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,10,???,AS???,100.0,1,1,0.00,0.00,0.00,0.00,0.0
